# 02 · Tokenização (SentencePiece BPE)

Inspeção dos tokenizadores SentencePiece BPE treinados para inglês e português (`data/tokenizer/spm_en.model`, `data/tokenizer/spm_pt.model`), vocabulário de 16.000 subpalavras cada.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src import config
from src.data.tokenizer import load_tokenizer, encode, decode

sp_en = load_tokenizer(config.SRC_LANG)
sp_pt = load_tokenizer(config.TGT_LANG)
print('vocab EN:', sp_en.get_piece_size())
print('vocab PT:', sp_pt.get_piece_size())

vocab EN: 16000
vocab PT: 16000


## Tokens especiais

In [2]:
for i in range(4):
    print(i, repr(sp_en.id_to_piece(i)))

0 '<pad>'
1 '<unk>'
2 '<s>'
3 '</s>'


## Exemplos de segmentação BPE

In [3]:
examples_en = [
    'The company needs to translate documents quickly.',
    'Internationalization and localization are important.',
    'How are you today?',
]
for text in examples_en:
    pieces = sp_en.encode(text, out_type=str)
    ids = encode(sp_en, text)
    print(f'{text!r}')
    print('  pieces:', pieces)
    print('  ids:   ', ids)
    print()

'The company needs to translate documents quickly.'
  pieces: ['▁The', '▁company', '▁needs', '▁to', '▁translate', '▁documents', '▁quickly', '.']
  ids:    [2, 90, 1812, 1569, 28, 14941, 3988, 3471, 15600, 3]

'Internationalization and localization are important.'
  pieces: ['▁International', 'ization', '▁and', '▁local', 'ization', '▁are', '▁important', '.']
  ids:    [2, 4253, 2582, 55, 2283, 2582, 126, 1024, 15600, 3]

'How are you today?'
  pieces: ['▁How', '▁are', '▁you', '▁today', '?']
  ids:    [2, 397, 126, 32, 987, 15616, 3]



## Encode/decode round-trip (português, com acentuação)

In [4]:
examples_pt = [
    'A empresa precisa traduzir documentos rapidamente.',
    'Internacionalização e localização são importantes.',
    'Como você está hoje?',
]
for text in examples_pt:
    ids = encode(sp_pt, text)
    reconstructed = decode(sp_pt, ids)
    print(f'original:      {text}')
    print(f'pieces:        {sp_pt.encode(text, out_type=str)}')
    print(f'reconstruído:  {reconstructed}')
    print()

original:      A empresa precisa traduzir documentos rapidamente.
pieces:        ['▁A', '▁empresa', '▁precisa', '▁traduz', 'ir', '▁documentos', '▁rapidamente', '.']
reconstruído:  A empresa precisa traduzir documentos rapidamente.

original:      Internacionalização e localização são importantes.
pieces:        ['▁Interna', 'cion', 'alização', '▁e', '▁localização', '▁são', '▁importantes', '.']
reconstruído:  Internacionalização e localização são importantes.

original:      Como você está hoje?
pieces:        ['▁Como', '▁você', '▁está', '▁hoje', '?']
reconstruído:  Como você está hoje?



## Estatística: nº médio de subpalavras por sentença (split de treino)

In [5]:
import numpy as np

with open(config.DATA_PROCESSED_DIR / 'train.en', encoding='utf-8') as f:
    en_lines = [line.strip() for line in f][:5000]
with open(config.DATA_PROCESSED_DIR / 'train.pt', encoding='utf-8') as f:
    pt_lines = [line.strip() for line in f][:5000]

en_token_counts = [len(sp_en.encode(t, out_type=int)) for t in en_lines]
pt_token_counts = [len(sp_pt.encode(t, out_type=int)) for t in pt_lines]

print(f'EN: média={np.mean(en_token_counts):.1f} subpalavras/sentença, máx={max(en_token_counts)}')
print(f'PT: média={np.mean(pt_token_counts):.1f} subpalavras/sentença, máx={max(pt_token_counts)}')

EN: média=11.7 subpalavras/sentença, máx=67
PT: média=11.2 subpalavras/sentença, máx=73
